## Example 1: Seismic Model Generation

This example demonstrates the use of **ModGen2D** to generate **multiple spatially correlated seismic property fields** for a typical **P-wave refraction scenario**. Refer **detailed version** for step-wise description.

### Model Replication and Automation
This **multiple version** of the example demonstrates how to generate multiple model realizations in a systematic and reproducible manner. For this and other examples, the individual cells from the **detailed version** can be consolidated into a single model-generation function. 

Multiple realizations can be generated efficiently by repeatedly calling the function with different model_id values by defining:
1. a random number generator (RNG) seed, and
2. a function parameterized by a model identifier (model_id),

### Notes on Random Number Generator (RNG) Usage

There are two common approaches for using RNGs to generate multiple realizations:

1. Single RNG instance (used in this example)
2. Independent RNG per realization, where rng = f(model_id) (recommended)

Using a single RNG instance can be memory-efficient, as the property configuration does not need to be redefined for each realization. However, regenerating a specific model (or a small subset of models) requires regenerating all preceding realizations.

In contrast, assigning an independent RNG to each realization allows individual models to be reproduced directly using their corresponding model_id. Since property configuration definitions are generally not memory-intensive, this approach is recommended for most use cases. For very large or highly complex configurations, a shared RNG may still be advantageous. The application of Method 2 is demonstrated in Example 2.

Overall, this approach enhances reproducibility, scalability, and code clarity, while preserving the same modeling logic presented in the detailed version.

In [1]:
import sys, importlib
sys.path.insert(0, r'F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator')

In [2]:
import pandas as pd
import numpy as np
import modgen2d as mg
import example1_helper_functions as hf

has_surface = True
layer0_flag=True
generate_obstacles=False
seed = 10  # Single seed for all realization generation (Method 1)

In [3]:
### ------------------------------------------------------------
### Step 1: Unit Configuration
units_config = mg.Units("m", "cm", conversion_factor=100) #mg.Units() also have same arg values

### ------------------------------------------------------------
### Step 2: Properties Definition
## Step2.1: General Definitions

# Domain dimensions
x_span = 5          # Domain length in X-direction (domain units)
z_span = 3          # Domain depth in Z-direction (domain units)

# Grid spacing
del_xz_final = 0.1       # Base spacing for the domain
del_xz_obs = del_xz_final / 10  # Finer spacing for obstacles (recommended: 1/10)

# Random number generator (for reproducibility): Use Model_id for different seed.
rng = np.random.default_rng(seed=seed)

# Interface interpolation method
remesh_interp_method = 'linear'

# Spatial correlation parameters
spatial_theta_x = 100  # Correlation length in X
spatial_theta_z = 0.5  # Correlation length in Z

## Step 2.2: Features Configuration
soil_props_csv = pd.read_csv('data/soil_properties.csv', index_col=0).apply(pd.to_numeric, errors='coerce')

# Initialize feature configuration
feature_config_instance =  mg.FeaturesConfig()

# Define material distributions
# For this example, we have multiple types of soil types and P(each type of soil) is same. 
soil_materials_distribution = mg.random_generators.DiscreteChoice(x = soil_props_csv.index.tolist(), rng=rng)

# Add features to feature_config_instance
feature_config_instance.add_feature('def', soil_materials_distribution, feature_description = 'def means soil.')


if generate_obstacles:
    # If there are utilities to add.
    utils_props_csv = pd.read_csv('data/utils_properties.csv', index_col=0).apply(pd.to_numeric, errors='coerce')
    utils_materials_distribution = mg.random_generators.DiscreteChoice(x = utils_props_csv.index.tolist(), rng=rng)
    feature_config_instance.add_feature('U', utils_materials_distribution, feature_description = 'utility features')


## Step 2.3: Main Properties
#2.3.1 Main Properties config definition
main_properties_config_instance = mg.MainPropertiesConfig(feature_config_instance, layer0_flag=layer0_flag)

cov_dc_ec = mg.random_generators.Constant(0.05, rng)
main_properties_info = {'vs': {'desc':'Shear wave velocity in m/s', 'cov': cov_dc_ec, 'layer0':[20, 20]}, 
                       'miu': {'desc':"Poisson's ratio", 'cov': None, 'layer0':[0, 0.4999]},  
                       'rho': {'desc':"Density", 'cov': cov_dc_ec, 'layer0':[1.2, 1000]}}



for main_property_name, main_props_vals in main_properties_info.items():
    #2.3.2 Define each MainProperty instance
    main_property_instance = mg.MainProperty(main_property_name, feature_config_instance, layer0_flag=layer0_flag, description=main_props_vals['desc'])

    #2.3.3  Define wet and dry properties for each features' each materials (including layer 0  for 'def' if flag is True)
    ## For Feature 'def'; material type 'all'
    main_property_instance = hf.add_features_from_pd(rng, main_property_instance, main_property_name, 'def', pd_dataframe=soil_props_csv, cov_distribution = main_props_vals['cov'], cov_type='cov')
    
    if generate_obstacles:
        main_property_instance = hf.add_features_from_pd(rng, main_property_instance, main_property_name, 'U', pd_dataframe=utils_props_csv, cov_distribution = main_props_vals['cov'], cov_type='cov')
    
    ## For Feature 'def'; material type 'layer0'
    if layer0_flag:
        air_val, water_val = main_props_vals['layer0']
        main_property_instance = hf.add_layer0(rng, main_property_instance, main_property_name, air_val, water_val)

    # 2.3.4 Add MainProperty to MainPropertiesConfig instance
    main_properties_config_instance.add_main_property(main_property_instance)

# main_properties_config_instance.print()

## Step 2.4: Auxiliary Properties
# Define some additional Properties
aux_props = mg.AuxillaryProperties()
aux_props.add_aux_property('n_layers',  mg.random_generators.DiscreteChoice(x=[1,2], p=[0.3, 0.7], rng=rng))
aux_props.add_aux_property('gwt', mg.random_generators.Uniform(0, z_span, rng))

if generate_obstacles:
    utilities_sett={
        'n_obs': mg.random_generators.DiscreteChoice([0,1,2,3], rng=rng),
        'obs_shape': mg.random_generators.DiscreteChoice(['circ2d', 'rect2d',], [1/2, 1/2], rng=rng),
        'dia_obs':mg.random_generators.Uniform(1, 3, rng=rng), 
        'lh_obs':mg.random_generators.Uniform(1, 3, rng=rng), 
        'obs_x_coord':mg.random_generators.Uniform(0, x_span, rng=rng), 
        'depth_top_dist':mg.random_generators.Discrete2ContinuousPDF([0,5], p = [1,.2], new_del_x = 1, rng=rng), #Continuous distribution: discrete to be converted into continuous: also
    }
    
    aux_props.add_aux_property('n_obs', utilities_sett['n_obs'])
    aux_props.add_aux_property('obs_shape', utilities_sett['obs_shape'])
    aux_props.add_aux_property('dia_obs', utilities_sett['dia_obs'])
    aux_props.add_aux_property('lh_obs', utilities_sett['lh_obs'])
    aux_props.add_aux_property('obs_x_coord', utilities_sett['obs_x_coord'])
    aux_props.add_aux_property('depth_top_utils', utilities_sett['depth_top_dist'])
    
# aux_props.print()

### ------------------------------------------------------------
### Step 3: Model Definition
## Step 3.1: 2D Domain Definition
domain_final = mg.DiscretizedDomain2D(x_span, z_span, del_xz_final, del_xz_final, units_config)

## Step 3.2: Interface Definitions
surface_factor = 1.5 if has_surface else 0

interface_sett= {
    'generate_surface':has_surface,  # Generate ground surface

    # Parameters for step 1: Generation of rough interfaces
    'rough_interface_creator_instance':mg.rough_interface_creator2d.UniformInterfaceGen(1, rng=rng),
    'rough_interface_generator_scale': [surface_factor,1.3,1.2,1], 
    # If number of layers > length of list, last value is reused

    # Parameters for step 2: Filtering
    'filter_settings': {
                 'filter_window_length':21, # must be odd
                 'filter_polyorder':7,
                        },

    # Parameter for Step 3: Interface Initial Points Generation
    'interfaces_depths_generation':'random', 
    'interfaces_depth_reference_point_x':None,
    
    # Parameter for Step 4: Handling the overlapping.
    'processing_settings': {
                'simulate_erosion': True,
            }
    }


In [4]:
for model_id in range(1,21):
    print(f"Generating {model_id}")
    
    ## Step 3.2: Interface Definitions (Continued)
    n_layers = aux_props.aux_properties['n_layers'].generate()
    gwt_depth = aux_props.aux_properties['gwt'].generate()
    
    # DiscretizedInterfaces2D from dictionary definition
    soil_interface = mg.DiscretizedInterfaces2DFromDict(domain_final, n_layers, interface_sett, remesh_interp_method=remesh_interp_method, rng=rng)
    # soil_interface.plot()
    
    ### Step 3.3: Lithological Domain (2D) Definition
    ## Step 3.3.1: From Interfaces (Global Soil Interface Configuration)
    
    # Reset global soil interface configuration (safety step)
    mg.GlobalSoilInterfaceConfig.reset()   # For safety only
    
    # Register soil interface
    mg.GlobalSoilInterfaceConfig.set_soil_interface(soil_interface)
    
    ## Get lithological domain from interface
    name = 'soil_lit'
    lit = mg.LithologicalDomain2D(domain_final, gwt_depth, name)
    # lit.plot(discrete_point_size = 10, plot_interfaces=True)
    
    ## Step 3.3.2: From Obstruction2D (Global Soil Interface Configuration)
    if generate_obstacles:
        ## Define a LithologicalDoamin2d From obstruction 2d.
        obs_lit = mg.LithologicalDomain2DFromObstruction2D(domain_final, 'obstructions')
        
        # Number of obstructions to generate
        n_obs = 0#aux_props.aux_properties['n_obs'].generate()
        
        # Randomly generate obstruction shapes
        obs_shapes = aux_props.aux_properties['obs_shape'].generate((n_obs,))
        
        # For each obstruction, create a Obstruction2D instance first, and then add to obs_lit. 
        for i, obs_shape in enumerate(obs_shapes):
            # Generate obstruction location
            obs_x_coord = aux_props.aux_properties['obs_x_coord'].generate()
            d_obs = aux_props.aux_properties['depth_top_utils'].generate()
        
            # Unique obstruction ID
            obs_id = i+1
        
            # Initialize Obstruction2D object
            obs_instance = mg.Obstruction2D(dl = del_xz_obs, ref_xz_symbolic = ['c', 0], snap_to_dl=True)
        
            # Define obstruction geometry
            if obs_shape == 'circ2d':
                d = aux_props.aux_properties['dia_obs'].generate()
                obs_instance.circle_2d(d, obstruction_id = obs_id)
            elif obs_shape == 'rect2d':
                lx = aux_props.aux_properties['lh_obs'].generate()
                lz = aux_props.aux_properties['lh_obs'].generate()
                obs_instance.rectangle_2d(lx, lz, obstruction_id = obs_id)
            else:
                raise ValueError(f"Invalid util_shape {obs_shape}")
            # obs_instance.plot()
        
            ## Add Obs2D into defnined lit_domain_from_obs2d 
            obs_lit.add_obstruction2D(obs_instance, shift_ref2d_to_xy = [obs_x_coord, d_obs], feature_id='U')
        
            # Plot lithological domain generated from obstructions (JUST FOR VISUALIZATION)
            # obs_lit.plot()
            
        
        # Visualize merged lithological domain (JUST FOR VISUALIZATION)
        merged_lit = lit.return_merged_lithological_domain([obs_lit], warn_if_null_lithological_domain=False)  # Just to see how merged lit. Karst to be added later.
        # merged_lit.plot()
    
    ## Step 3.4: Lithological Domain Collection
    
    # Initialize lithological domain collection
    lit_collection = mg.LithologicalDomain2DCollection(main_properties_config_instance.get_feature_ids(), interface_set_name="soil") 
    
    # Add soil-based lithological domain
    lit_collection.add_lithological_domain_from_soil_interface_config(lit)
    
    if generate_obstacles:
        lit_collection.add_lithological_domain_from_obstruction2d("obs", obs_lit, warn_if_null_lithological_domain=False)
    # Finalize and lock the lithological domain collection
    lit_collection.lock()
    
    ### ------------------------------------------------------------
    ### Step 4: Generate Simulated Property Profiles
    # Unlock main properties config to allow sampling
    main_properties_config_instance.unlock()
    
    # Sample property values for all features in the lithological domain
    main_properties_config_instance.lock_and_generate_sample_properties(lit_collection)
    
    # Initialize spatial simulator (controls spatial correlation)
    spatial_sim = mg.spatial_simulator2d.CovarianceDecompositionSimulator(spatial_theta_x, spatial_theta_z, rng = rng)
    
    # Generate property profiles using the spatial simulator
    gen_profiles = mg.GeneratedProfileCollection2D(main_properties_config_instance, lit_collection, spatial_sim)
    gen_profiles.simulate_zvals_property_profile('z_vals_seismic')
    gen_profiles.simulate_profile_from_zvals_property_profile('vs', 'z_vals_seismic')     
    gen_profiles.simulate_profile_from_zvals_property_profile('rho', 'z_vals_seismic')     
    gen_profiles.simulate_profile_from_zvals_property_profile('miu', 'z_vals_seismic')           
    
    #Save the profiles into h5 files
    i = model_id
    gen_profiles.save_to_hdf5(f'generated_h5_files/{i:08d}.h5', hdf5_compression_level=8)

Generating 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Data saved to generated_h5_files/00000001.h5
Generating 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 0 (wet): sigma=0 but 18 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 220 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:360: UserWarning: Layer 2 (dry): sigma=88.88616365066214 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 1 (wet): sigma=0 but 389 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 1 (dry): sigma=0 but 80 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulato

Data saved to generated_h5_files/00000002.h5
Generating 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Data saved to generated_h5_files/00000003.h5
Generating 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 0 (wet): sigma=0 but 146 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 45 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:307: UserWarning: Layer 2: sigma=0 but 1309 z-values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 319 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 1 (wet): sigma=0 but 200 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning:

Data saved to generated_h5_files/00000004.h5
Generating 5
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Data saved to generated_h5_files/00000005.h5
Generating 6
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1


...

F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 89 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 1 (wet): sigma=0 but 250 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 1 (dry): sigma=0 but 1161 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 349 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:360: UserWarning: Layer 2 (dry): sigma=89.73762322565993 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simula

Data saved to generated_h5_files/00000015.h5
Generating 16
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Data saved to generated_h5_files/00000016.h5
Generating 17
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 0 (wet): sigma=0 but 131 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 43 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 1 (wet): sigma=0 but 1319 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 1 (dry): sigma=0 but 7 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 168 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:349: UserWarnin

Data saved to generated_h5_files/00000017.h5
Generating 18
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Data saved to generated_h5_files/00000018.h5
Generating 19
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Data saved to generated_h5_files/00000019.h5
Generating 20
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 0
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Data saved to generated_h5_files/00000020.h5


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 0 (wet): sigma=0 but 106 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 398 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:345: UserWarning: Layer 1 (wet): sigma=0 but 894 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 1 (dry): sigma=0 but 102 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:356: UserWarning: Layer 0 (dry): sigma=0 but 127 values are non-zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\2D_Model_Generator\modgen2d\spatial_simulator2d.py:307: UserWarn